# Notebook Test

In [7]:
# import

from IPython.display import Video
import cv2
import easyocr
# from paddleocr import PaddleOCR
import matplotlib.pyplot as plt
from scipy.stats import skew, kurtosis

from sklearn.cluster import KMeans

import pandas as pd
import numpy as np
from tqdm import tqdm

In [3]:
user = "USER_alpedhuez"
video_id = "VIDEO_6814175723704683782"
video_path = f"videos_mp4/downloads/{user}/{video_id}.mp4"
Video(video_path, width=250)

In [30]:
# load dataset

df_train = pd.read_csv("X_train.csv", sep=';')
df_test = pd.read_csv("X_test.csv", sep=';')
df_train.head()

,id,album,artist,artists,aspect_ratio,channel,description,video_duration,format,release_year,track,uploader,filepath,download_timing,uploader_short,vid,uid
0,7602656035161050390,NaN,Urhov Bogdan,['Urhov Bogdan'],0.56,Davos Klosters,you dream you 🥹 #davosklosters #skiing #mounta...,13,1080x1920,2026,оригинальный звук,davosklosters,downloads/USER_davosklosters/VIDEO_76026560351...,2026-02-12 09:29:25,davos,VIDEO_7602656035161050390,USER_davosklosters
1,7590718903144287510,NaN,LykTraffx,['LykTraffx'],0.56,Davos Klosters,already missing this again 🥹 #spenglercup #dav...,12,1080x1920,2026,оригинальный звук,davosklosters,downloads/USER_davosklosters/VIDEO_75907189031...,2026-02-12 09:29:25,davos,VIDEO_7590718903144287510,USER_davosklosters
2,7571821778746592534,NaN,ALTÉGO,['ALTÉGO'],0.56,Davos Klosters,how??!!!🥹🥹 #davosklosters #skiing #ski #season...,13,1080x1920,2025,THE FATE OF OPHELIA X MIDNIGHT CITY,davosklosters,downloads/USER_davosklosters/VIDEO_75718217787...,2026-02-12 09:29:25,davos,VIDEO_7571821778746592534,USER_davosklosters
3,7569927329154190614,NaN,Raye,['Raye'],0.56,Davos Klosters,We're back on snow!🥹🎿 #davosklosters #seasonop...,7,1080x1920,2025,original sound,davosklosters,downloads/USER_davosklosters/VIDEO_75699273291...,2026-02-12 09:29:25,davos,VIDEO_7569927329154190614,USER_davosklosters
4,7566270741134462230,NaN,músicas e traduções,['músicas e traduções'],0.56,Davos Klosters,"Somebody send help, pls. 🥶 #davos #firstsnow #...",8,1080x1920,2025,som original,davosklosters,downloads/USER_davosklosters/VIDEO_75662707411...,2026-02-12 09:29:25,davos,VIDEO_7566270741134462230,USER_davosklosters


# Extraction d'images avec OpenCV

In [3]:
cap = cv2.VideoCapture(video_path)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    
    # Ici, 'frame' est une image (matrice NumPy)
    # Vous pouvez calculer des features ici (ex: luminosité moyenne)
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
cap.release()

# Extraction des couleurs principales

In [ ]:
import cv2
import numpy as np
from sklearn.cluster import KMeans

def extract_dominant_colors(video_path, n_clusters=3, sample_rate=30):
    cap = cv2.VideoCapture(video_path)
    pixels = []

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        # On ne prend qu'une frame sur 30 (environ 1 par seconde)
        if int(cap.get(cv2.CAP_PROP_POS_FRAMES)) % sample_rate == 0:
            # On redimensionne pour accélérer le calcul (le clustering est gourmand)
            small_frame = cv2.resize(frame, (56, 100))
            # OpenCV utilise BGR, on repasse en RGB pour une interprétation humaine simple
            rgb_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB) # image RGB de taille (height, width, 3)
            pixels.append(rgb_frame.reshape(-1, 3)) # listes de tous les pixels Red, Green, Blue
            
    cap.release()

    # On regroupe tous les pixels échantillonnés
    all_pixels = np.vstack(pixels)

    # Application de K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=5)
    kmeans.fit(all_pixels)

    # Récupération des couleurs (centres des clusters)
    colors = kmeans.cluster_centers_.astype(int)
    
    # Optionnel : Calculer le pourcentage de chaque couleur
    labels = kmeans.labels_
    counts = np.bincount(labels)
    percentages = counts / len(labels)

    return colors, percentages

# Utilisation
video_id = "VIDEO_6821547365887905030"
video_path = f"videos_mp4/downloads/{user}/{video_id}.mp4"

top_colors, weights = extract_dominant_colors(video_path)

print("Couleurs dominantes (RGB) :")
for i, color in enumerate(top_colors):
    print(f"Couleur {i+1}: {color} - Présence: {weights[i]*100:.1f}%")

Couleurs dominantes (RGB) :
Couleur 1: [116 139 157] - Présence: 31.6%
Couleur 2: [167 187 206] - Présence: 62.6%
Couleur 3: [43 46 39] - Présence: 5.7%


In [15]:
df = df_test

df_colors = pd.DataFrame(columns=["video_id", 'R1', 'G1', 'B1', 'W1', 'R2', 'G2', 'B2', 'W2', 'R3', 'G3', 'B3', 'W3'])

for path, vid in tqdm(zip(df['filepath'], df['vid']), total = len(df), desc="Recherche Couleurs Principales"):
    # calcul des couleurs principales
    top_colors, color_weights = extract_dominant_colors(f"videos_mp4/{path}", n_clusters=3, sample_rate=60)
    
    new_lign = pd.DataFrame([{
        'video_id': vid, 
        'R1': top_colors[0][0],
        'G1': top_colors[0][1],
        'B1': top_colors[0][2],
        'W1' : color_weights[0],
        'R2': top_colors[1][0],
        'G2': top_colors[1][1],
        'B2': top_colors[1][2],
        'W2' : color_weights[1],
        'R3': top_colors[2][0],
        'G3': top_colors[2][1],
        'B3': top_colors[2][2],
        'W3' : color_weights[2]
    }])
    df_colors = pd.concat([df_colors, new_lign], ignore_index=True)


Recherche Couleurs Principales:   0%|          | 0/338 [00:00<?, ?it/s]/tmp/ipykernel_212456/1343376478.py:24: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_colors = pd.concat([df_colors, new_lign], ignore_index=True)
Recherche Couleurs Principales: 100%|██████████| 338/338 [09:28<00:00,  1.68s/it]  


In [16]:
df_colors.head()

df_colors.to_csv("top_colors_test.csv", sep=",")

# Détection des textes

In [37]:
# On initialise le moteur une seule fois hors de la fonction pour gagner du temps
# gpu=True si vous avez une carte graphique (ou sur Google Colab)
reader = easyocr.Reader(['fr', 'en'], gpu=False)


def extract_text_from_video(video_path):
    """
    Analyse seulement 3 images clés : 1s après début, milieu, 1s avant fin.
    """
    cap = cv2.VideoCapture(video_path)

    # --- Étape 1 : Calculer les positions (en frames) ---
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS)
    
    # 1 seconde après le début, au milieu, 1 seconde avant la fin
    # On utilise max/min pour éviter les erreurs sur les vidéos trop courtes
    pos_debut = int(min(fps, total_frames * 0.1))
    pos_milieu = int(total_frames / 2)
    pos_fin = int(max(0, total_frames - fps))
    
    target_frames = [pos_debut, pos_milieu, pos_fin]
    
    all_detected_text = []

    # --- Étape 2 : Boucle sur les 3 images cibles ---
    for target in target_frames:
        # On déplace le "curseur" de la vidéo à la frame choisie
        cap.set(cv2.CAP_PROP_POS_FRAMES, target)
        ret, frame = cap.read()
        
        if not ret:
            continue
            
        # Redimensionnement (640px de large)
        h, w = frame.shape[:2]
        new_w = 640
        new_h = int(h * (new_w / w))
        img_resized = cv2.resize(frame, (new_w, new_h))

        # FILTRE 1 : Noir et Blanc (Niveaux de gris)
        gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)

        # 2c. FILTRE 2 : Thresholding d'Otsu
        # On utilise THRESH_BINARY_INV pour avoir le texte en BLANC sur fond NOIR
        # (c'est le format que préfère souvent EasyOCR)
        # S'il rate certains textes, essayez THRESH_BINARY (texte noir sur fond blanc).
        _, img_thresholded = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        
        # OCR (detail=0 pour n'avoir que le texte brut)
        results = reader.readtext(img_thresholded, detail=1) # On garde detail=1 pour la confiance
        
        for res in results:
            text = res[1]
            conf = res[2]
            if conf > 0.4:
                all_detected_text.append(text.lower().strip())
    
    cap.release()

    # --- Étape 3 : Nettoyage ---
    unique_text = sorted(list(set([t for t in all_detected_text if len(t) > 2])))
    full_string = " | ".join(unique_text)
    
    return {
        "has_text": len(unique_text) > 0,
        "text_raw": full_string,
        "nb_textbloc": len(unique_text)
    }

Using CPU. Note: This module is much faster with a GPU.


In [36]:
# test du code

video_id = "VIDEO_7604886753471761686"
video_path = f"videos_mp4/downloads/{user}/{video_id}.mp4"

dict = extract_text_from_video(video_path)

print(f"Result : {dict['has_text']}, {dict['text_raw']}, {dict['nb_textbloc']}")

NameError: name 'extract_text_from_video' is not defined

In [38]:
df = df_train

df_text_train = pd.DataFrame(columns=["video_id", 'nb_textbloc'])

for path, vid in tqdm(zip(df['filepath'], df['vid']), total = len(df), desc="Recherche Text"):
    # extraction text
    results = extract_text_from_video(f"videos_mp4/{path}")
    
    new_lign = pd.DataFrame([{
        'video_id': vid, 
        'nb_textbloc': results['nb_textbloc']
    }])
    df_text_train = pd.concat([df_text_train, new_lign], ignore_index=True)


df = df_test

for path, vid in tqdm(zip(df['filepath'], df['vid']), total = len(df), desc="Recherche Text"):
    # extraction text
    results = extract_text_from_video(f"videos_mp4/{path}")
    
    new_lign = pd.DataFrame([{
        'video_id': vid, 
        'nb_textbloc': results['nb_textbloc']
    }])
    df_text_test = pd.concat([df_text_test, new_lign], ignore_index=True)


Recherche Text:   0%|          | 0/1348 [00:05<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
df_text_train.to_csv("text_train.csv", sep=",")

df_text_test.to_csv("text_test.csv", sep=",")

,video_id,nb_words
0,VIDEO_7602656035161050390,1
1,VIDEO_7590718903144287510,3
2,VIDEO_7571821778746592534,2
3,VIDEO_7569927329154190614,0
4,VIDEO_7566270741134462230,11
5,VIDEO_7536170254989315350,2
6,VIDEO_7520640970199698710,0
7,VIDEO_7496799936558796054,0
8,VIDEO_7487960173529582870,1
9,VIDEO_7478712216724802838,1


# Carte Attention

In [41]:
def saliency_video(video_path, plot=False):
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_interval = int(fps / 2)

    saliency = cv2.saliency.StaticSaliencyFineGrained_create()
    scores = []
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break
        
        if frame_count % frame_interval == 0:
            # 1. Redimensionner pour accélérer le calcul
            frame_small = cv2.resize(frame, (160, 284)) # (480, 854)

            # 2. CALCUL
            success, saliency_map = saliency.computeSaliency(frame_small)

            saliency_img = (saliency_map * 255).astype("uint8")
            
            if plot:
                heatmap = cv2.applyColorMap(saliency_img, cv2.COLORMAP_JET)
                frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                heatmap_rgb = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
                # 4. Affichage avec Matplotlib
                plt.figure(figsize=(12, 6))
                
                plt.subplot(1, 2, 1)
                plt.imshow(frame_rgb)
                plt.title("Vidéo Originale (TikTok)")
                plt.axis('off')

                plt.subplot(1, 2, 2)
                plt.imshow(heatmap_rgb)
                plt.title("Saliency Map (Heatmap)")
                plt.axis('off')

                plt.tight_layout()
                plt.show()

            if success:
                # Score de concentration (écart-type)
                score = np.std(saliency_map) * 100
                scores.append(score)
            else:
                scores.append(0)

        frame_count += 1

    cap.release()

    return(scores)


def extract_attention_features(scores):
    scores = np.array(scores)
    # On ignore le premier score s'il est à 0 (souvent un artefact de chargement)
    clean_scores = scores[scores > 0] if len(scores[scores > 0]) > 0 else scores

    features = {
        # --- Stats de base ---
        "mean_saliency": np.mean(clean_scores),
        "std_saliency": np.std(clean_scores),
        "max_saliency": np.max(clean_scores),
        "min_saliency": np.min(clean_scores),
        
        # --- Forme de la distribution ---
        "skewness": skew(clean_scores), # Asymétrie (pics au début ou à la fin)
        "kurtosis": kurtosis(clean_scores), # "Pointu" (pics isolés) vs "Plat" (stable)
        
        # --- Rythme TikTok ---
        "hook_score": np.mean(scores[:3]), # Qualité du début (les 1.5 premières sec)
        "outro_score": np.mean(scores[-3:]), # Qualité de la fin
        "peak_location": np.argmax(scores) / len(scores), # Position relative du pic (0 à 1)
        
        # --- Seuils de performance ---
        "high_attention_ratio": np.sum(scores > 13) / len(scores),
        "attention_instability": np.mean(np.abs(np.diff(scores))), # Variation moyenne entre frames
    }
    
    return features

In [44]:
# --- TEST ---
user = "USER_davosklosters" # "USER_alpedhuez"
video_id = "VIDEO_7602656035161050390" # "VIDEO_6814175723704683782"
path = f"videos_mp4/downloads/{user}/{video_id}.mp4"
scores = saliency_video(path, plot=False)
features = extract_attention_features(scores)

print(scores)
print(features)

[np.float32(24.90884), np.float32(24.350962), np.float32(23.841566), np.float32(23.087486), np.float32(26.366695), np.float32(25.08537), np.float32(24.450586), np.float32(23.732664), np.float32(23.50564), np.float32(22.881851), np.float32(23.170036), np.float32(23.859686), np.float32(22.060307), np.float32(23.059353), np.float32(24.035046), np.float32(22.52939), np.float32(22.369448), np.float32(20.593412), np.float32(22.51555), np.float32(20.959229), np.float32(20.598986), np.float32(20.105236), np.float32(21.026695), np.float32(22.295643), np.float32(21.11946), np.float32(22.60101), np.float32(22.024122), np.float32(22.883705)]
{'mean_saliency': np.float32(22.857784), 'std_saliency': np.float32(1.4753263), 'max_saliency': np.float32(26.366695), 'min_saliency': np.float32(20.105236), 'skewness': np.float32(0.12388368), 'kurtosis': np.float32(-0.3303573), 'hook_score': np.float32(24.367125), 'outro_score': np.float32(22.502945), 'peak_location': np.float64(0.14285714285714285), 'high_a

In [ ]:
# df = df_train

# df_att_train = pd.DataFrame(columns=['video_id', 'att_mean_saliency', 'att_max_saliency', 
#                                       'att_min_saliency', 'att_skewness', 'att_kurtosis', 
#                                       'att_hook_score', 'att_outro_score', 'att_peak_location', 
#                                       'att_high_attention_ratio', 'att_attention_instability'])


# for path, vid in tqdm(zip(df['filepath'], df['vid']), total = len(df), desc="Recherche Text"):
#     # extraction text
#     scores = saliency_video(f"videos_mp4/{path}")
#     features = extract_attention_features(scores)
    
#     new_lign = pd.DataFrame([{
#         'video_id': vid, 
#         'att_mean_saliency' : float(features['mean_saliency']), 
#         'att_max_saliency' : float(features['max_saliency']), 
#         'att_min_saliency' : float(features['min_saliency']), 
#         'att_skewness' : float(features['skewness']), 
#         'att_kurtosis' : float(features['kurtosis']), 
#         'att_hook_score' : float(features['hook_score']), 
#         'att_outro_score' : float(features['outro_score']), 
#         'att_peak_location' : float(features['peak_location']), 
#         'att_high_attention_ratio' : float(features['high_attention_ratio']), 
#         'att_attention_instability' : float(features['attention_instability'])
#     }])
#     df_att_train = pd.concat([df_att_train, new_lign], ignore_index=True)

# df_att_train.to_csv("attention_train.csv", sep=",")


df = df_test

df_att_test = pd.DataFrame(columns=['video_id', 'att_mean_saliency', 'att_max_saliency', 
                                      'att_min_saliency', 'att_skewness', 'att_kurtosis', 
                                      'att_hook_score', 'att_outro_score', 'att_peak_location', 
                                      'att_high_attention_ratio', 'att_attention_instability'])

for path, vid in tqdm(zip(df['filepath'], df['vid']), total = len(df), desc="Recherche Text"):
    # extraction text
    scores = saliency_video(f"videos_mp4/{path}")
    features = extract_attention_features(scores)
    
    new_lign = pd.DataFrame([{
        'video_id': vid, 
        'att_mean_saliency' : float(features['mean_saliency']), 
        'att_max_saliency' : float(features['max_saliency']), 
        'att_min_saliency' : float(features['min_saliency']), 
        'att_skewness' : float(features['skewness']), 
        'att_kurtosis' : float(features['kurtosis']), 
        'att_hook_score' : float(features['hook_score']), 
        'att_outro_score' : float(features['outro_score']), 
        'att_peak_location' : float(features['peak_location']), 
        'att_high_attention_ratio' : float(features['high_attention_ratio']), 
        'att_attention_instability' : float(features['attention_instability'])
    }])

    df_att_test = pd.concat([df_text_test, new_lign], ignore_index=True)

    df_att_test.to_csv("attention_test.csv", sep=",")


Recherche Text:   0%|          | 0/1348 [00:00<?, ?it/s]/tmp/ipykernel_176048/3125103433.py:27: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_att_train = pd.concat([df_att_train, new_lign], ignore_index=True)
Recherche Text: 100%|██████████| 1348/1348 [10:43<00:00,  2.09it/s]


In [ ]:
df_att_train.to_csv("attention_train.csv", sep=",", index=False)
df_att_train.head()

,video_id,att_mean_saliency,att_max_saliency,att_min_saliency,att_skewness,att_kurtosis,att_hook_score,att_outro_score,att_peak_location,att_high_attention_ratio,att_attention_instability
0,VIDEO_7602656035161050390,22.857784,26.366695,20.105236,0.123884,-0.330357,24.367125,22.502945,0.142857,1.000000,1.014669
1,VIDEO_7590718903144287510,18.023708,20.543884,15.062449,-0.221365,-0.762860,18.653322,20.109137,0.962963,1.000000,1.109325
2,VIDEO_7571821778746592534,16.916769,19.434734,12.808052,-0.895561,-0.095923,12.877566,16.231546,0.785714,0.857143,0.726305
3,VIDEO_7569927329154190614,20.543476,22.149172,18.173630,-0.853218,-0.335328,20.707413,19.448420,0.000000,1.000000,0.983737
4,VIDEO_7566270741134462230,17.050192,19.879618,12.307085,-0.700854,-0.578920,17.201696,18.656586,0.833333,0.833333,1.237883
